In [1]:
import transformers
from transformers import (
    MT5ForConditionalGeneration,
    Seq2SeqTrainer, MT5Tokenizer, MT5Config
)
import json
import datasets
import pandas as pd
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
import numpy as np
from datasets import load_metric
import gc
import datasets
import os
import torch


os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID" # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["WANDB_DISABLED"] = "true"
!export CUDA_VISIBLE_DEVICES=0

device, use_gpu = ("cuda:0", True) if torch.cuda.is_available() else ("cpu", False)

2025-06-11 04:04:50.006610: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749614690.022585   12875 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749614690.027616   12875 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-11 04:04:50.043369: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
import boto3
import json


s3 = boto3.client("s3")
bucket = "hiep-delta-bk"

full_data = []

for num in range(0, 200):
    key = f"chunks_out_c/chunk_{num:04d}.json"

    obj = s3.get_object(Bucket=bucket, Key=key)
    data = json.load(obj['Body'])

    [full_data.append(item) for item in data]

casual_train = [{"src": item["source"], "tgt": item["1"]} for item in full_data]
coarse_train = [{"src": item["source"], "tgt": item["2"]} for item in full_data]
formal_train = [{"src": item["source"], "tgt": item["3"]} for item in full_data]
chinese_train = [{"src": item["source"], "tgt": item["4"]} for item in full_data]


In [3]:
checkpoint = "VietAI/vit5-base"
model = MT5ForConditionalGeneration.from_pretrained(checkpoint)
print('load model done')
tokenizer = MT5Tokenizer.from_pretrained(checkpoint)
print('load tokenizer done')

You are using a model of type t5 to instantiate a model of type mt5. This is not supported for all configurations of models and can yield errors.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'T5Tokenizer'. 
The class this function is called from is 'MT5Tokenizer'.
You are using the default legacy behaviour of the <class 'transformers.models.mt5.tokenization_mt5.MT5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


load model done
load tokenizer done


In [4]:
tdata = pd.DataFrame(formal_train)
tdata = tdata.reset_index()
dataset = datasets.Dataset.from_pandas(tdata)

train = dataset.train_test_split(
    test_size=2800, seed=42
)

train_data = train['train']
test_data = train['test']

def format_dataset(example):
     return {'input': example['src'], 'target': example['tgt']}
train_data = train_data.map(format_dataset, remove_columns=train_data.column_names)
test_data = test_data.map(format_dataset, remove_columns=test_data.column_names)

def convert_to_features(example_batch):
    input_encodings = tokenizer.batch_encode_plus(example_batch['input'], pad_to_max_length=True, max_length=128)
    target_encodings = tokenizer.batch_encode_plus(example_batch['target'], pad_to_max_length=True, max_length=128)
    encodings = {
        'input_ids': input_encodings['input_ids'], 
        'attention_mask': input_encodings['attention_mask'],
        'labels': target_encodings['input_ids'],
        'decoder_attention_mask': target_encodings['attention_mask']
    }

    return encodings
train_data = train_data.map(convert_to_features, batched=True, remove_columns=train_data.column_names)
test_data = test_data.map(convert_to_features, batched=True, remove_columns=test_data.column_names)

Parameter 'function'=<function format_dataset at 0x7fcf3533ac00> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


  0%|          | 0/194054 [00:00<?, ?ex/s]

  0%|          | 0/2800 [00:00<?, ?ex/s]

  0%|          | 0/195 [00:00<?, ?ba/s]

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
/opt/conda/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:2700: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


  0%|          | 0/3 [00:00<?, ?ba/s]

In [5]:
from datasets import load_metric
rouge = load_metric("rouge")

def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions

    # all unnecessary tokens are removed
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    labels_ids[labels_ids == -100] = tokenizer.pad_token_id
    label_str = tokenizer.batch_decode(labels_ids, skip_special_tokens=True)

    rouge_output = rouge.compute(predictions=pred_str, references=label_str, rouge_types=["rouge2"])["rouge2"].mid

    return {
        "rouge2_precision": round(rouge_output.precision, 4),
        "rouge2_recall": round(rouge_output.recall, 4),
        "rouge2_fmeasure": round(rouge_output.fmeasure, 4),
    }

In [6]:
data_collator = DataCollatorForSeq2Seq(tokenizer,model=model)
training_args = Seq2SeqTrainingArguments(
    output_dir="viT5_f_thesis_chinese",
    per_device_train_batch_size=16,
    num_train_epochs=2,
    per_device_eval_batch_size=16,
    predict_with_generate=True,
    eval_strategy="steps",
    do_train=True,
    do_eval=True,
    logging_steps=22859,
    save_strategy="steps",
    save_steps=45718,
    eval_steps=22859,
    overwrite_output_dir=True,
    save_total_limit=4,
    load_best_model_at_end=True,
    report_to=None,
     group_by_length=True,
    #fp16=True, 
)
trainer = Seq2SeqTrainer(
    model=model,
    data_collator = data_collator,
    tokenizer = tokenizer,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_data,
    eval_dataset=test_data,
)


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipykernel_12875/3856144613.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [7]:
trainer.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss,Validation Loss,Rouge2 Precision,Rouge2 Recall,Rouge2 Fmeasure
22859,0.401200,0.181315,0.687500,0.462600,0.533400


TrainOutput(global_step=24258, training_loss=0.3887330170898035, metrics={'train_runtime': 7408.8723, 'train_samples_per_second': 52.384, 'train_steps_per_second': 3.274, 'total_flos': 6.734828799472435e+16, 'train_loss': 0.3887330170898035, 'epoch': 2.0})

In [10]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
 
model = AutoModelForSeq2SeqLM.from_pretrained('./<model>/checkpoint-24258')
tokenizer = AutoTokenizer.from_pretrained('./<model>/checkpoint-24258')
 
def generate_summary(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True).to(model.device)
    summary_ids = model.generate(**inputs, max_length=256, num_beams=5, early_stopping=True)
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)
 
 
sample_text = """
TEXT ~~~
"""
output = generate_summary(sample_text)
print("👉 Output:", output)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'MT5Tokenizer'. 
The class this function is called from is 'T5Tokenizer'.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


👉 Output: Bác sĩ Kee Yuan từ Bệnh viện Đại học quốc gia Singapore cũng thừa nhận kỹ thuật mổ nội soi tuyến giáp của các bác sĩ Việt Nam có nhiều ưu điểm vượt trội so với các quốc gia và vùng lãnh thổ khác trên thế giới.
